# dNaty — Experimentos Completos (Google Colab)
> **Dynamic Neuro-Adaptive sYstem with evoluTionarY learning**  
> Config completa: G=50, N=20, T_local=5, dataset completo (60K MNIST)  
> Tempo estimado: ~25 min com GPU T4

**O que este notebook faz:**
1. Instala dependências
2. Clona/sobe o código dNaty
3. Roda Experimento 1: MNIST + FashionMNIST (5 seeds)
4. Roda Experimento 3: Split-MNIST Continual Learning (5 seeds)
5. Gera `DNATY_RESULTS.md` com todos os resultados reais
6. Faz download automático dos JSONs de resultado

## Célula 1 — Verificar GPU

In [ ]:
import torch
print('GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('AVISO: sem GPU — mude Runtime > Change runtime type > T4 GPU')

## Célula 2 — Instalar dependências

In [ ]:
!pip install torch torchvision numpy scipy matplotlib tqdm -q
print('Dependências instaladas')

## Célula 3 — Upload do código dNaty
> Faça upload do arquivo `dnaty_colab.zip` (zipar a pasta `dnaty/` do seu projeto)  
> **OU** cole o código diretamente nas células abaixo (opção automática)

In [ ]:
from google.colab import files
import zipfile, os

print('Selecione o arquivo dnaty_code.zip...')
uploaded = files.upload()

for fname in uploaded:
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('.')
    print(f'Descompactado: {fname}')

os.makedirs('results', exist_ok=True)
os.makedirs('data', exist_ok=True)
print('Pronto!')


## Célula 4 — Módulo: EpisodicMemory

In [ ]:
%%writefile dnaty/core/__init__.py


In [ ]:
%%writefile dnaty/core/memory.py
from __future__ import annotations
from dataclasses import dataclass, field
import numpy as np

@dataclass
class Experience:
    operator: str
    delta_loss: float
    gradient_norm: float
    generation: int
    weight: float = 1.0
    timestamp: int = 0

    @property
    def impact(self) -> float:
        if self.delta_loss >= 0:
            return 0.0
        return abs(self.delta_loss) * self.gradient_norm

class EpisodicMemory:
    def __init__(self, max_size=1000, decay_gamma=0.99):
        self.experiences = []
        self.max_size = max_size
        self.gamma = decay_gamma
        self._step = 0

    def update(self, exp):
        for e in self.experiences:
            e.weight *= self.gamma
        exp.timestamp = self._step
        self._step += 1
        self.experiences.append(exp)
        if len(self.experiences) > self.max_size:
            self._prune()

    def _prune(self):
        self.experiences.sort(
            key=lambda e: e.weight*(self.gamma**max(0,self._step-e.timestamp)),
            reverse=True)
        self.experiences = self.experiences[:self.max_size]

    def query_mutation_probs(self, operators, tau=1.0):
        scores = {op: 0.0 for op in operators}
        for exp in self.experiences:
            if exp.operator in scores and exp.impact > 0:
                t_decay = self.gamma**max(0, self._step-exp.timestamp)
                scores[exp.operator] += exp.impact * t_decay
        vals = np.array([scores[op] for op in operators], dtype=np.float64)/max(tau,1e-8)
        vals -= vals.max()
        exp_vals = np.exp(vals)
        probs = exp_vals/exp_vals.sum()
        return {op: float(p) for op, p in zip(operators, probs)}

    def operator_counts(self, operators):
        counts = {op: 0 for op in operators}
        for e in self.experiences:
            if e.operator in counts and e.impact > 0:
                counts[e.operator] += 1
        return counts


## Célula 5 — Módulo: Arquitetura (DynamicMLP)

In [ ]:
%%writefile dnaty/core/arch.py
from __future__ import annotations
import torch, torch.nn as nn
import numpy as np
from copy import deepcopy

ACTIVATIONS = {'relu':nn.ReLU,'tanh':nn.Tanh,'gelu':nn.GELU,'sigmoid':nn.Sigmoid}
_innov = 0
def next_innovation():
    global _innov; _innov+=1; return _innov

class DynamicMLP(nn.Module):
    def __init__(self, layer_sizes, activations=None, n_classes=10):
        super().__init__()
        self.layer_sizes = list(layer_sizes)
        self.n_classes = n_classes
        self.activations = activations or ['relu']*(len(layer_sizes)-1)
        self.innovation_ids = [next_innovation() for _ in range(len(layer_sizes)-1)]
        self._build()

    def _build(self):
        layers = []
        for i in range(len(self.layer_sizes)-1):
            layers.append(nn.Linear(self.layer_sizes[i], self.layer_sizes[i+1]))
            act = self.activations[i] if i < len(self.activations) else 'relu'
            layers.append(ACTIVATIONS.get(act, nn.ReLU)())
        layers.append(nn.Linear(self.layer_sizes[-1], self.n_classes))
        self.net = nn.Sequential(*layers)
        self.skip_connections = []

    def forward(self, x):
        x = x.view(x.size(0), -1)
        layer_outputs = [x]
        idx = 0
        for i in range(len(self.layer_sizes)-1):
            out = self.net[idx+1](self.net[idx](layer_outputs[-1]))
            for src,dst,proj in self.skip_connections:
                if dst==i+1 and src<len(layer_outputs):
                    si = layer_outputs[src]
                    if proj is not None: si=proj(si)
                    if si.shape==out.shape: out=out+si
            layer_outputs.append(out)
            idx+=2
        return self.net[idx](layer_outputs[-1])

    def count_params(self): return sum(p.numel() for p in self.parameters())
    def count_flops(self):
        f=0
        for i in range(len(self.layer_sizes)-1): f+=2*self.layer_sizes[i]*self.layer_sizes[i+1]
        return f+2*self.layer_sizes[-1]*self.n_classes
    def is_valid(self): return all(s>0 for s in self.layer_sizes) and len(self.layer_sizes)>=2


In [ ]:
%%writefile dnaty/core/individual.py
from copy import deepcopy
from dnaty.core.arch import DynamicMLP
from dnaty.core.memory import EpisodicMemory

class Individual:
    def __init__(self, model, memory=None):
        self.model = model
        self.memory = memory or EpisodicMemory()
        self.last_op = 'init'
        self.fitness = (0.,0.,0.)
        self.acc = 0.
        self.last_grad_norm = 0.

    def clone(self):
        ind = Individual(deepcopy(self.model), deepcopy(self.memory))
        ind.last_op=self.last_op; ind.fitness=self.fitness; ind.acc=self.acc
        return ind

    def count_params(self): return self.model.count_params()
    def count_flops(self): return self.model.count_flops()


import os, sys
sys.path.insert(0, '.')

files_needed = [
    'dnaty/operators/mutations.py',
    'dnaty/evolution/selection.py',
    'dnaty/evolution/evolver.py',
    'dnaty/training/local_train.py',
    'dnaty/experiments/data_utils.py',
    'dnaty/experiments/baselines.py',
    'dnaty/experiments/exp1_mnist.py',
    'dnaty/experiments/exp3_cl.py',
    'dnaty/analysis/cl_metrics.py',
    'dnaty/analysis/stats.py',
    'dnaty/analysis/report.py',
]
missing = [f for f in files_needed if not os.path.exists(f)]
if missing:
    print('ERRO — arquivos faltando:')
    for m in missing: print(' ', m)
    print('\nVolte na Celula 3 e faca o upload do dnaty_code.zip')
else:
    print('OK — todos os arquivos presentes, pode continuar!')
    # Teste rapido de import
    from dnaty.core.memory import EpisodicMemory
    from dnaty.core.arch import DynamicMLP
    print('Imports OK!')


In [ ]:
# Escrever operators/__init__.py
open('dnaty/operators/__init__.py','w').close()
open('dnaty/evolution/__init__.py','w').close()
open('dnaty/training/__init__.py','w').close()
open('dnaty/experiments/__init__.py','w').close()
open('dnaty/analysis/__init__.py','w').close()
open('dnaty/__init__.py','w').close()
print('__init__.py criados')

In [ ]:
# Copiar arquivos do projeto local para o Colab
# Se fez upload do zip, já estão disponíveis.
# Caso contrário, os arquivos serão criados pelas células acima.
import os
files_needed = [
    'dnaty/operators/mutations.py',
    'dnaty/evolution/selection.py',
    'dnaty/evolution/evolver.py',
    'dnaty/training/local_train.py',
    'dnaty/experiments/data_utils.py',
    'dnaty/experiments/baselines.py',
    'dnaty/experiments/exp1_mnist.py',
    'dnaty/experiments/exp3_cl.py',
    'dnaty/analysis/cl_metrics.py',
    'dnaty/analysis/stats.py',
    'dnaty/analysis/report.py',
]
missing = [f for f in files_needed if not os.path.exists(f)]
if missing:
    print('FALTANDO (fazer upload do zip):', missing)
else:
    print('Todos os arquivos presentes!')

## Célula 7 — Configuração dos Experimentos
> Ajuste aqui para config completa do paper ou config rápida

In [ ]:
# CONFIG: completa = G=50, N=20, T_local=5, dataset completo
# CONFIG: rapida   = G=15, N=6,  T_local=2, subset 3K

CONFIG = 'completa'  # mude para 'rapida' para teste rapido

if CONFIG == 'completa':
    N_GENERATIONS = 50
    N_POP = 20
    T_LOCAL = 5
    TRAIN_SUBSET = None   # 60K amostras completas
    VAL_SUBSET = None     # 10K test completo
    print('Config COMPLETA: G=50, N=20, T_local=5, 60K amostras')
    print('Tempo estimado: ~45 min com GPU T4')
else:
    N_GENERATIONS = 15
    N_POP = 6
    T_LOCAL = 2
    TRAIN_SUBSET = 3000
    VAL_SUBSET = 1000
    print('Config RAPIDA: G=15, N=6, T_local=2, 3K amostras')
    print('Tempo estimado: ~5 min com GPU T4')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


## Célula 8 — Experimento 1: MNIST + FashionMNIST

In [ ]:
import sys, os
sys.path.insert(0, '.')

# Sobrescrever configurações nos módulos
import dnaty.experiments.exp1_mnist as exp1_mod
exp1_mod.N_GENERATIONS = N_GENERATIONS
exp1_mod.N_POP = N_POP
exp1_mod.T_LOCAL = T_LOCAL
exp1_mod.TRAIN_SUBSET = TRAIN_SUBSET
exp1_mod.VAL_SUBSET = VAL_SUBSET

print('Iniciando Experimento 1...')
results_exp1 = exp1_mod.main()
print('Experimento 1 concluído!')

## Célula 8b — Experimento 2: CIFAR-10 (operadores convolucionais reais)

In [ ]:
import dnaty.experiments.exp2_cifar as exp2_mod
exp2_mod.N_GENERATIONS = N_GENERATIONS
exp2_mod.N_POP = N_POP
exp2_mod.T_LOCAL = T_LOCAL
exp2_mod.TRAIN_SUBSET = TRAIN_SUBSET if TRAIN_SUBSET else 10000
exp2_mod.VAL_SUBSET = VAL_SUBSET if VAL_SUBSET else 2000

print('Iniciando Experimento 2 (CIFAR-10)...')
results_exp2 = exp2_mod.main()
print('Experimento 2 concluido!')

## Célula 9 — Experimento 3: Split-MNIST Continual Learning

In [ ]:
import dnaty.experiments.exp3_cl as exp3_mod
exp3_mod.N_EPOCHS_CL = 10 if CONFIG=='completa' else 5
exp3_mod.TRAIN_SUBSET_CL = None if CONFIG=='completa' else 800

print('Iniciando Experimento 3 (Continual Learning)...')
results_exp3 = exp3_mod.main()
print('Experimento 3 concluído!')

## Célula 10 — Gerar Relatório e Download

In [ ]:
from google.colab import files
files.download('results/exp1_results.json')
files.download('results/exp2_cifar10_results.json')
files.download('results/exp3_cl_results.json')
files.download('DNATY_RESULTS.md')
print('Download iniciado!')

In [ ]:
# Download dos resultados
from google.colab import files
files.download('results/exp1_results.json')
files.download('results/exp3_cl_results.json')
files.download('DNATY_RESULTS.md')
print('Download iniciado!')

## Célula 11 — Gráfico de Convergência (opcional)
> Visualizar a convergência diretamente no Colab

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as mstyle
import numpy as np

plt.style.use('dark_background')
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.patch.set_facecolor('#05070f')

colors = {'MNIST':'#4f8fff', 'FashionMNIST':'#50e0a0'}

for idx, ds in enumerate(['MNIST','FashionMNIST']):
    ax_acc = axes[0][idx]
    ax_delta = axes[1][idx]
    ax_acc.set_facecolor('#080c18')
    ax_delta.set_facecolor('#080c18')

    # Seed 0 history
    hist = e1[ds]['dnaty'][0]['history']
    gens = [h['gen'] for h in hist]
    acc = [h['best_acc']*100 for h in hist]
    dg = [h['delta_grad'] for h in hist]
    dm = [h['delta_mem'] for h in hist]

    # Accuracy
    ax_acc.plot(gens, acc, color=colors[ds], linewidth=2.5, marker='o', markersize=4)
    ax_acc.axhline(y=97.3 if ds=='MNIST' else 89.1, color='#f0c060', linestyle='--', alpha=0.5, label='Meta paper')
    ax_acc.set_title(f'{ds} — Acurácia (seed=0)', color='#d8e4f8', fontsize=11)
    ax_acc.set_ylabel('Acurácia (%)', color='#8898bb')
    ax_acc.tick_params(colors='#4a5a7a')
    ax_acc.legend(fontsize=8)
    ax_acc.grid(alpha=0.1)

    # Deltas
    ax_delta.plot(gens, dg, color='#50e0a0', linewidth=2, marker='o', markersize=3, label='δ_grad')
    ax_delta.plot(gens, dm, color='#f0c060', linewidth=2, marker='o', markersize=3, label='δ_mem')
    ax_delta.axhline(y=0, color='#50e0a0', linestyle='--', alpha=0.4, label='≥ 0 (Teorema 1)')
    ax_delta.set_title(f'{ds} — δ_grad e δ_mem (Teorema 1)', color='#d8e4f8', fontsize=11)
    ax_delta.set_ylabel('δ', color='#8898bb')
    ax_delta.set_xlabel('Geração', color='#8898bb')
    ax_delta.tick_params(colors='#4a5a7a')
    ax_delta.legend(fontsize=8)
    ax_delta.grid(alpha=0.1)

plt.suptitle('dNaty — Convergência Real · Validação Teorema 1', color='#d8e4f8', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('dnaty_convergence.png', dpi=150, bbox_inches='tight', facecolor='#05070f')
plt.show()
files.download('dnaty_convergence.png')
print('Gráfico salvo!')

---
## Instruções de uso

1. **Abrir no Colab:** `File > Upload notebook` ou arrastar este `.ipynb`
2. **Ativar GPU:** `Runtime > Change runtime type > T4 GPU > Save`
3. **Fazer upload do código:** zipar a pasta `dnaty/` do projeto e fazer upload na Célula 3
4. **Rodar tudo:** `Runtime > Run all`
5. **Baixar resultados:** Célula 10 faz download automático dos JSONs e do `.md`
6. **Substituir no projeto local:** copiar os JSONs para `results/` e rodar `python dnaty/analysis/report.py`

**Após rodar no Colab com config completa, os resultados esperados são:**
- MNIST: ~97.3% com ~63K params
- FashionMNIST: ~89.1% com ~68K params  
- Split-MNIST BWT: ~−0.031 (vs EWC −0.089)
